In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

baseline_files = [
    "baseline_gaf.npy", "baseline_rp.npy", "baseline_mtf.npy",
    "baseline_heatmap.npy", "baseline_radar.npy",
]

In [2]:
baseline_meta = pd.read_parquet(PROCESSED_DIR / "baseline_metadata.parquet")
print("baseline_metadata shape:", baseline_meta.shape)
print(baseline_meta.head())

baseline_metadata shape: (576848, 5)
  code_module code_presentation  id_student  week  label
0         AAA             2013J       11391     0      0
1         AAA             2013J       11391     1      0
2         AAA             2013J       11391     2      0
3         AAA             2013J       11391     4      0
4         AAA             2013J       11391     5      0


In [3]:
for fname in baseline_files:
    arr = np.load(PROCESSED_DIR / fname, mmap_mode="r")
    sample = arr[:1000]
    print(f"{fname}: shape={arr.shape}, has_nan={np.isnan(sample).any()}, "
          f"min={sample.min():.3f}, max={sample.max():.3f}")

baseline_gaf.npy: shape=(576848, 20, 7, 7), has_nan=False, min=-1.000, max=1.000
baseline_rp.npy: shape=(576848, 20, 7, 7), has_nan=False, min=0.000, max=0.000
baseline_mtf.npy: shape=(576848, 20, 7, 7), has_nan=False, min=0.000, max=1.000
baseline_heatmap.npy: shape=(576848, 4, 5), has_nan=False, min=0.000, max=1.000
baseline_radar.npy: shape=(576848, 64, 64), has_nan=False, min=0.000, max=1.000


In [4]:
print(baseline_meta.shape)
print(baseline_meta["label"].value_counts(normalize=True))

(576848, 5)
label
0    0.90951
1    0.09049
Name: proportion, dtype: float64


In [5]:
ati_meta = pd.read_parquet(PROCESSED_DIR / "ati_v1_metadata.parquet")

print("ATI-v1 label distribution:")
print(ati_meta["label"].value_counts(normalize=True))
print("\nBaseline label distribution:")
print(baseline_meta["label"].value_counts(normalize=True))

print("\nSố dòng ATI-v1:", len(ati_meta), " | Số dòng baseline:", len(baseline_meta))

merged_check = ati_meta.merge(
    baseline_meta, on=["code_module", "code_presentation", "id_student", "week"],
    suffixes=("_ati", "_baseline"), how="outer", indicator=True
)
print("\nKhớp hoàn toàn cả 2 bảng (both):", (merged_check["_merge"] == "both").sum())
print("Chỉ có ở ATI-v1:", (merged_check["_merge"] == "left_only").sum())
print("Chỉ có ở baseline:", (merged_check["_merge"] == "right_only").sum())
print("Nhãn lệch nhau giữa 2 bảng:",
      (merged_check["label_ati"] != merged_check["label_baseline"]).sum())

ATI-v1 label distribution:
label
0    0.90951
1    0.09049
Name: proportion, dtype: float64

Baseline label distribution:
label
0    0.90951
1    0.09049
Name: proportion, dtype: float64

Số dòng ATI-v1: 576848  | Số dòng baseline: 576848

Khớp hoàn toàn cả 2 bảng (both): 576848
Chỉ có ở ATI-v1: 0
Chỉ có ở baseline: 0
Nhãn lệch nhau giữa 2 bảng: 0
